In [0]:
import asyncio
import httpx

GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"

async def geocode(
    client: httpx.AsyncClient, name: str
):
    r = await client.get(
        GEOCODING_URL,
        params={"name": name, "count": 10, "language": "en", "format": "json"},
    )
    return name, r.status_code, r.json().get("results") or []

async def fetchall(names: list[str]):
    async with httpx.AsyncClient(timeout=30) as client:
        tasks = [geocode(client, name) for name in names]
        return await asyncio.gather(*tasks, return_exceptions=True)


city_dim_table = "workspace.open_meteo.dim_city"
city_names = ["London", "Paris", "Rome", "Madrid", "Berlin", "Moscow", "New York"]

results = await fetchall(city_names)

rows = []
for item in results:
    if isinstance(item, BaseException):
        # TODO: Log the failure — this city silently drops
        continue
    name, status_code, candidates = item
    if status_code != 200 or not candidates:
        # TODO: Log the bad status / empty match — this city silently drops
        continue
    result = max(candidates, key=lambda c: c.get("population") or 0)
    rows.append((
        result["id"],
        result["name"],
        result["country"],
        result["latitude"],
        result["longitude"],
        result["elevation"],
    ))

spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {city_dim_table} (
    city_id    INT,
    name       STRING,
    country    STRING,
    latitude   DOUBLE,
    longitude  DOUBLE,
    elevation  DOUBLE
  )
""")

spark.createDataFrame(
    rows,
    "city_id INT, name STRING, country STRING, latitude DOUBLE, longitude DOUBLE, elevation DOUBLE",
).createOrReplaceTempView("city_staging")

spark.sql(f"""
  MERGE INTO {city_dim_table} AS t
  USING city_staging AS s
  ON t.city_id = s.city_id
  WHEN MATCHED THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
""")
    